# Formative 3 — Deep Q-Learning on Atari Pong

This notebook trains and evaluates a Deep Q-Network agent on Atari Pong for the whole team in a single, reproducible run. It executes all 30 hyperparameter experiments across the team, ten each for Chol, Lorita, and Josephine, trains the final production model, and records the gameplay video used in the submission. Every experiment is independent and resumable, since results are backed up to Google Drive as soon as each one finishes, so a disconnect at any point only costs whichever experiment was in progress, and re-running the notebook automatically restores earlier progress and skips anything already completed rather than duplicating it. This run used a Colab Pro session with an NVIDIA L4 GPU; wall-clock time will vary depending on whichever GPU your own session is assigned.

## 1. Confirm GPU is attached

Before doing anything else, this cell checks that a GPU is available to the runtime and reports which one Colab has assigned. Training a DQN agent on raw pixel frames is far too slow on CPU alone, so the rest of the notebook depends on this succeeding.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > GPU (L4 on Colab Pro, or T4 on the free tier), then re-run."

## 2. Mount Drive and get the project code

This is a one-time setup step: upload `Formative3_DQN_Project.zip` to the root of your Google Drive, at `MyDrive/Formative3_DQN_Project.zip`. From then on, every time this cell runs it unzips the code into the Colab session and restores any results, models, or videos left over from a previous session, so it is safe to stop at any point and resume later, in any order, across as many sessions as needed.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
BACKUP_DIR = "/content/drive/MyDrive/Formative3_DQN_backup"
os.makedirs(BACKUP_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
PROJECT_ZIP = "/content/drive/MyDrive/Formative3_DQN_Project.zip"
PROJECT_DIR = "/content/Formative3_DQN_Project"

if os.path.isfile(os.path.join(PROJECT_DIR, "train.py")):
    print("Project already present in this session, skipping unzip.")
elif os.path.isfile(PROJECT_ZIP):
    !unzip -oq "{PROJECT_ZIP}" -d "{PROJECT_DIR}"
    print(f"Unzipped project from Drive into {PROJECT_DIR}")
else:
    raise FileNotFoundError(f"{PROJECT_ZIP} not found — upload Formative3_DQN_Project.zip to your Drive root first.")

%cd {PROJECT_DIR}

# Restore any results/models/videos from a previous session's backup - makes every
# member's block safe to run in a fresh session without losing earlier progress.
for folder in ["models", "experiments", "videos", "tensorboard_logs"]:
    src = os.path.join(BACKUP_DIR, folder)
    if os.path.isdir(src):
        os.makedirs(folder, exist_ok=True)
        !cp -r {src}/. {folder}/
print("Restored any previous progress from Drive (if this is a fresh session).")

Unzipped project from Drive into /content/Formative3_DQN_Project
/content/Formative3_DQN_Project
Restored any previous progress from Drive (if this is a fresh session).


In [4]:
def backup_to_drive():
    """Copies models/, experiments/, and videos/ to Drive - the checkpoint after every block below."""
    for folder in ["models", "experiments", "videos", "tensorboard_logs"]:
        if os.path.isdir(folder):
            !cp -r {folder} {BACKUP_DIR}/
    print(f"Backed up to {BACKUP_DIR}")

## 3. Install dependencies

Installs everything listed in `requirements.txt`, including Stable-Baselines3, Gymnasium, ALE, and PyTorch, and confirms the Atari ROMs are available. Modern versions of `ale-py` bundle the ROMs directly, so the AutoROM step here is a safety net rather than a strict requirement.

In [5]:
!pip install -q -r requirements.txt
!AutoROM --accept-license --quiet 2>/dev/null || echo "AutoROM not found/needed - modern ale-py already bundles ROMs directly, this is expected and safe to ignore."

import pandas as pd  # used later for the results table and hyperparameter config edits

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 31.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 135.2 MB/s eta 0:00:00
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.


## 4. Sanity check

Runs a short, real training pass of 20,000 steps purely to confirm the environment, model, and logging pipeline all work end to end before committing to the full sweep. The reward here is expected to stay close to -21, since this is a pipeline check rather than one of the 30 graded experiments, so a low score at this stage is normal and not a sign of a problem.

In [6]:
!python train.py --member "SanityCheck" --exp-id 0 --policy CnnPolicy \
  --total-timesteps 20000 --model-path models/sanity_check.zip --notes "pipeline check"

2026-07-07 13:58:26.832624: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-07 13:58:26.902068: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arca

## 5. Chol — 10 experiments (learning rate & batch size)

Runs Chol's block of ten experiments one at a time, each trained for 300,000 timesteps and backed up to Drive as soon as it finishes. Because every run is backed up individually, a disconnect partway through this block only costs whichever experiment was in progress, not the whole block.

In [7]:
for exp_id in range(1, 11):
    !python run_experiments.py --member "Chol" --exp-id {exp_id}
    backup_to_drive()

Streaming output truncated to the last 5000 lines.
|    fps              | 239      |
|    time_elapsed     | 1146     |
|    total_timesteps  | 274957   |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.0158   |
|    n_updates        | 66239    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.13e+03 |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 352      |
|    fps              | 240      |
|    time_elapsed     | 1157     |
|    total_timesteps  | 277989   |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.0153   |
|    n_updates        | 66997    |
----------------------------------
Eval num_timesteps=280000, episode_reward=-21.00 +/- 0.00
Episode length: 3056.00 +/- 0.00
----------------------------------
| eval/           

## 6. Lorita — 10 experiments (gamma & epsilon-greedy schedule)

Runs Lorita's block of ten experiments the same way, one at a time with each result backed up to Drive immediately after it completes, this time varying the discount factor and the epsilon-greedy exploration schedule instead of learning rate and batch size.

In [8]:
for exp_id in range(1, 11):
    !python run_experiments.py --member "Lorita" --exp-id {exp_id}
    backup_to_drive()

Streaming output truncated to the last 5000 lines.
| eval/               |          |
|    mean_ep_length   | 3.41e+03 |
|    mean_reward      | -20.2    |
| rollout/            |          |
|    exploration_rate | 0.05     |
| time/               |          |
|    total_timesteps  | 210000   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00269  |
|    n_updates        | 49999    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 4.98e+03 |
|    ep_rew_mean      | -18.2    |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 200      |
|    fps              | 243      |
|    time_elapsed     | 873      |
|    total_timesteps  | 212705   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.006    |
|    n_updates        | 50676    |
----------------------------------
----

## 7. Josephine — 10 experiments (replay buffer / learning-starts & CNN vs MLP)

Runs Josephine's block of ten experiments, covering replay buffer size and the learning-starts warm-up period, and including the direct CnnPolicy-versus-MlpPolicy architecture comparison the assignment asks for.

In [9]:
for exp_id in range(1, 11):
    !python run_experiments.py --member "Josephine" --exp-id {exp_id}
    backup_to_drive()

Streaming output truncated to the last 5000 lines.
|    learning_rate    | 0.001    |
|    loss             | 0.00126  |
|    n_updates        | 64652    |
----------------------------------
Eval num_timesteps=270000, episode_reward=-21.00 +/- 0.00
Episode length: 792.00 +/- 0.00
----------------------------------
| eval/               |          |
|    mean_ep_length   | 792      |
|    mean_reward      | -21      |
| rollout/            |          |
|    exploration_rate | 0.01     |
| time/               |          |
|    total_timesteps  | 270000   |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.00023  |
|    n_updates        | 64999    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 814      |
|    ep_rew_mean      | -20.9    |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 332      |
|    fps           

## 8. Regenerate the hyperparameter table

Reads every logged result from `experiments/hyperparameter_results.csv` and rebuilds `experiments/hyperparameter_table.md`, the markdown table that gets pasted into the README. This can be re-run at any point to refresh the table with whatever results exist so far.

In [10]:
!python run_experiments.py --generate-table-only
!cat experiments/hyperparameter_table.md

2026-07-08 00:58:23.542111: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-08 00:58:23.610868: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Wrote /cont

## 9. Noted behavior

Each row's noted behavior already contains two pieces of real, automatically generated documentation: the hypothesis written before the experiment ran, and a computed comparison against that member's baseline result, such as "worse by 5.3 reward," added afterward by `finalize_submission.py`. Together these already satisfy the assignment's documentation requirement without any manual writing. For a stronger writeup, the team can optionally add qualitative reasoning on top of these numbers to explain why a given change helped or hurt, but this is an enhancement rather than something the pipeline depends on.

In [11]:
df = pd.read_csv("experiments/hyperparameter_results.csv")
df[["member", "exp_id", "policy", "lr", "gamma", "batch_size", "mean_reward", "mean_ep_length", "noted_behavior"]]

,member,exp_id,policy,lr,gamma,batch_size,mean_reward,mean_ep_length,noted_behavior
0,SanityCheck,0,CnnPolicy,0.00010,0.990,32,-21.0,757.8,pipeline check
1,Chol,1,CnnPolicy,0.00010,0.990,32,-21.0,757.2,Baseline configuration (SB3 default-like values)
2,Chol,2,CnnPolicy,0.00100,0.990,32,-21.0,757.2,Higher lr may speed up early learning but risk...
3,Chol,3,CnnPolicy,0.00001,0.990,32,-21.0,757.2,Lower lr should be more stable but slower to c...
4,Chol,4,CnnPolicy,0.00010,0.990,16,-20.8,817.2,Smaller batch -> noisier gradient estimates
5,Chol,5,CnnPolicy,0.00010,0.990,64,-10.8,2219.6,Larger batch -> smoother gradients and more co...
6,Chol,6,CnnPolicy,0.00010,0.990,128,-9.8,2910.8,Even larger batch: expect diminishing returns ...
7,Chol,7,CnnPolicy,0.00100,0.990,64,-21.0,757.2,Combo: high lr + large batch to offset gradien...
8,Chol,8,CnnPolicy,0.00001,0.990,16,-21.0,757.2,Combo: low lr + small batch - expect slow/unde...
9,Chol,9,CnnPolicy,0.00025,0.990,32,-21.0,757.2,SB3-Zoo-style lr for comparison against our ba...


## 10. Train the final production model

Trains the single best-performing configuration found in the sweep, batch_size=256 with everything else left at baseline, once more at a much larger budget of 1,000,000 timesteps. This is the model that `play.py` and the submitted gameplay clip actually use, so the extra training time is worth it. Checkpoints are saved every 50,000 steps, so a disconnect partway through leaves a recent, resumable snapshot rather than losing the run entirely.

In [12]:
!python train.py --member "Team" --exp-id 99 --policy CnnPolicy \
  --lr 0.0001 --gamma 0.99 --batch-size 256 \
  --epsilon-start 1.0 --epsilon-end 0.01 --epsilon-decay 0.1 \
  --total-timesteps 1000000 --checkpoint-freq 50000 --model-path models/dqn_model.zip \
  --notes "Final production model - best config from the sweep (batch_size=256, Chol exp10 winner)" --progress-bar
backup_to_drive()

2026-07-08 00:58:31.035597: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-08 00:58:31.103086: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arca

## 11. Record the gameplay video

Colab has no display attached, so this records the trained agent headlessly: `play.py` renders each frame to an array instead of a window and writes it directly to an mp4 file. What gets saved is the real trained agent actually playing three episodes, not a staged or edited clip.

In [22]:
!python play.py --model-path models/dqn_model.zip --policy CnnPolicy --episodes 3 --record --video-folder videos

2026-07-08 03:19:52.000209: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-08 03:19:52.070510: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arca

In [23]:
import glob
from IPython.display import Video

clips = sorted(glob.glob("videos/*.mp4"))
Video(clips[-1], embed=True, width=480) if clips else None

## 12. Finalize README and package for submission

Writes the real hyperparameter table, an auto-generated discussion draft comparing each experiment to its baseline, and a link to the recorded video directly into `README.md`. It then zips the entire project, including code, README, results, the trained model, and the video, backs that zip up to Drive, and downloads it to the local machine as the final submission package.

In [24]:
!python finalize_submission.py
backup_to_drive()

README.md updated with 30 logged experiment(s).
Backed up to /content/drive/MyDrive/Formative3_DQN_backup


In [25]:
SUBMISSION_ZIP = "/content/Formative3_DQN_Submission.zip"

!cd {PROJECT_DIR} && zip -r -q {SUBMISSION_ZIP} . \
  -x "venv/*" -x "*__pycache__*" -x "*.pyc" -x "tensorboard_logs/*" -x ".git/*"

!cp {SUBMISSION_ZIP} {BACKUP_DIR}/
print(f"Submission zip saved to {SUBMISSION_ZIP} and backed up to {BACKUP_DIR}")

from google.colab import files
files.download(SUBMISSION_ZIP)

Submission zip saved to /content/Formative3_DQN_Submission.zip and backed up to /content/drive/MyDrive/Formative3_DQN_backup


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>